# Track designer

Draw a track as a rough polygon, smooth it into a drivable loop, preview it,
and install it — after `install_track` the track works everywhere a track
name is accepted (experiments, `rollout_video`, the CLI).

The same machinery imports any of the 126 official DeepRacer tracks:
`fetch_official_track("Vegas_track")`.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from deepracer_genesis.tools.track_builder import (
    build_route, build_track_mesh, install_track, fetch_official_track,
)

## 1. Sketch the centerline

List the corner points of your track in order (meters, any closed shape —
it closes automatically). Tweak, re-run, repeat. `half_width` is the road
half-width: the DeepRacer car is ~0.2 m wide; official tracks use ~0.53 m.

In [ ]:
points = [
    (0.0, 0.0),
    (6.0, 0.0),
    (8.0, 2.0),
    (6.0, 4.0),
    (4.0, 2.5),
    (2.0, 4.5),
    (-1.0, 3.0),
]
half_width = 0.53

route = build_route(points, half_width, n_waypoints=150, smooth_passes=3)
center, inner, outer = route[:, 0:2], route[:, 2:4], route[:, 4:6]

fig, ax = plt.subplots(figsize=(9, 6))
ax.fill(np.append(outer[:, 0], outer[0, 0]), np.append(outer[:, 1], outer[0, 1]),
        color="#2a2c33")
ax.fill(np.append(inner[:, 0], inner[0, 0]), np.append(inner[:, 1], inner[0, 1]),
        color="#5c8a5c")
ax.plot(np.append(center[:, 0], center[0, 0]), np.append(center[:, 1], center[0, 1]),
        "--", color="orange", lw=1.5)
ax.plot(*np.array(points + points[:1]).T, "o:", color="tomato", ms=5, alpha=0.6,
        label="your corner points")
ax.set_aspect("equal"); ax.legend(); ax.set_title(
    f"track length ~{np.linalg.norm(np.diff(np.vstack([center, center[:1]]), axis=0), axis=1).sum():.1f} m")
plt.show()

## 2. Install it

Writes `route.npy` + a generated road mesh (asphalt / border lines / dashed
centerline as tiny solid textures — renders identically under Madrona, Nyx
and the rasterizer) under `assets/tracks/generated/<name>/` and registers
the name. Generated tracks are re-discovered automatically in every process.

In [ ]:
install_track("my_track", route)
from deepracer_genesis.envs.track import TRACKS
print(sorted(TRACKS))

## 3. Sanity-drive it (subprocess — Genesis builds one scene per process)

A privileged P-controller drives 4 cars for 10 s and renders the bird's-eye
view. If the cars lap without leaving the road, the track is well-formed
(no self-intersections, radii the car can steer).

In [ ]:
%%writefile /tmp/_drive_my_track.py
import torch, genesis as gs
import imageio.v2 as imageio
gs.init(backend=gs.cuda, logging_level="warning")
from deepracer_genesis.configs.cfgs import get_env_cfg
from deepracer_genesis.envs import DeepRacerEnv

cfg = get_env_cfg(vision=False, track="my_track")
cfg.update(spectator=True, spectator_res=(960, 720))
env = DeepRacerEnv(num_envs=4, env_cfg=cfg)
prog = torch.zeros(4, device=env.device)
for _ in range(300):
    lat = env.lateral * env.dir_sign / env.half_width.clamp(min=0.1)
    steer = (-(1.1 * lat + 0.9 * torch.sin(env.heading_err))).clamp(-1, 1)
    env.step(torch.stack([steer, torch.full_like(steer, -0.3)], dim=1))
    prog += env.d_progress
imageio.imwrite("/tmp/my_track_topdown.png", env.render_spectator())
print("mean progress:", round(prog.mean().item(), 1), "m in 10 s |",
      "offtrack events:", int(env.offtrack_buf.sum().item()))

In [ ]:
!python /tmp/_drive_my_track.py
from IPython.display import Image
Image("/tmp/my_track_topdown.png", width=720)

## 4. Train on it

Any experiment takes the new name — single-file style:

```python
from deepracer_genesis.experiment import Experiment, FeatureEnvironment, VectorPolicy, run

class MyTrackRacer(Experiment):
    total_env_steps = 5_000_000
    eval_every_steps = 1_000_000
    def pipeline(self):
        return FeatureEnvironment(num_envs=1024, tracks=("my_track",)) >> VectorPolicy()

run(MyTrackRacer)
```

...or watch a policy trained elsewhere drive it:
`rollout_video("feature_baseline", track="my_track")`.